# 15 — Correcting the unit of supervision

Notebook 12 shipped a **review-level** model trained on star ratings. It scored
0.6746 accuracy on the holdout, which looked acceptable. It is not.

Probing it with clean prose exposed the problem:

| sentence | predicted |
|---|---|
| *"The room was spotless and the staff could not have been more helpful."* | **negative** |
| *"Arrived late, checked in, went to sleep. Nothing special."* | **negative** |

### Why

The reconstructed "review" text is a concatenation of aspect-matched sentence
fragments, and the label comes from the parent review's star rating. So the
training data contains rows like:

> `negative` — *"position to reach the airport staff"*
> `negative` — *"Good location near train station. Mould in bathroom…"*

Fragments with no sentiment at all, and fragments that open with praise, all
labelled by a rating that belongs to the whole review. The model learned
**which words co-occur with low ratings**, not what sentiment language looks
like. The holdout score is real but it measures the wrong thing: it rewards
exploiting the same star-correlation on held-out rows of the same corpus.

This is the failure the project's own evaluation contract is meant to catch,
and a per-comment accuracy number alone did not catch it. A handful of probe
sentences did.

### The fix

Supervise at the **sentence** level, where the label describes the text in
front of it, then roll sentence predictions up to a review by majority vote.
The sentence labels are the file's machine-generated ones — noisier per item,
but they describe the right unit. The rollup is then scored against the **real
star rating**, which the model never sees.

Both options measured on the same held-out reviews:

| | review macro-F1 | clean-prose probe |
|---|---|---|
| review-level + star labels | 0.6258 | 6/8 |
| **sentence-level + rollup** | **0.6990** | **7/8** |

In [1]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
from src.normalize import normalize, NORMALIZE_VERSION
OUT = REPO / "data_en"
SRC = REPO / "dataset for training in ENGLISH" / "df_final_absa_v4.csv"
LABELS = ["negative", "neutral", "positive"]

from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
import joblib

en = pd.read_csv(SRC)
en = en[en.langue == "Anglais"].copy()
en["phrase"] = en.phrase.astype(str).map(normalize)
en = en[en.phrase.str.len() > 0]

KEY = ["hotel_name", "date_sejour", "note", "type_voyage"]
en["rid"] = en.groupby(KEY, dropna=False).ngroup()      # stable review id
print(f"sentences {len(en):,}   reviews {en.rid.nunique():,}")

sentences 32,625   reviews 6,426


## Split by review, never by sentence

Sentences from one review must not straddle a split — they share vocabulary,
a hotel and a rating, so a sentence-level split would leak.

In [2]:
def star3(n):
    return "negative" if n < 3 else ("neutral" if n <= 3.5 else "positive")

rev = (en.groupby("rid")
         .agg(note=("note", "first"), hotel=("hotel_name", "first"),
              n_sent=("phrase", "size"))
         .reset_index())
rev["label"] = rev.note.map(star3)

tr_r, tmp_r = train_test_split(rev, test_size=.4, stratify=rev.label, random_state=42)
va_r, te_r = train_test_split(tmp_r, test_size=.5, stratify=tmp_r.label, random_state=42)
ca_r, ho_r = train_test_split(va_r, test_size=.5, stratify=va_r.label, random_state=42)

split = {}
for nm, d in [("train", tr_r), ("calib", ca_r), ("hold", ho_r), ("test", te_r)]:
    for r in d.rid: split[r] = nm
en["split"] = en.rid.map(split)

print(f"{'split':<8} {'reviews':>8} {'sentences':>10}")
for nm, d in [("train", tr_r), ("calib", ca_r), ("hold", ho_r), ("test", te_r)]:
    print(f"{nm:<8} {len(d):>8,} {(en.split==nm).sum():>10,}")

split     reviews  sentences
train       3,855     19,422
calib         642      3,011
hold          643      3,328
test        1,286      6,864


In [3]:
s_tr = en[en.split == "train"]
print(f"training on {len(s_tr):,} sentences")
print("sentence label mix:", dict(s_tr.sentiment.value_counts()))

t0 = time.perf_counter()
model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=3000, class_weight="balanced"),
).fit(s_tr.phrase, s_tr.sentiment)
print(f"trained in {time.perf_counter()-t0:.1f}s   classes {list(model.classes_)}")

training on 19,422 sentences
sentence label mix: {'positive': np.int64(10062), 'neutral': np.int64(5563), 'negative': np.int64(3797)}


trained in 1.6s   classes ['negative', 'neutral', 'positive']


## Two evaluations, because there are two units

**Sentence level** is scored against the file's machine labels — useful as a
consistency check, but those labels are only ~84% reliable, so it is a ceiling
on measurement, not on truth.

**Review level** is scored against the **star rating**, which is a genuine
human label the model never saw at any point. That is the number that counts.

In [4]:
CLS = list(model.classes_)
ORDER = [CLS.index(l) for l in LABELS]

def rollup(df, pred):
    """Majority vote of sentence predictions, per review."""
    return (pd.DataFrame({"rid": df.rid.values, "p": pred})
              .groupby("rid").p.agg(lambda s: s.value_counts().idxmax()))

for nm, rdf in [("calib", ca_r), ("hold", ho_r)]:
    s = en[en.split == nm]
    p = model.predict(s.phrase)
    print(f"[{nm}] sentence vs machine labels: acc={accuracy_score(s.sentiment,p):.4f}")
    r = rollup(s, p)
    gold = rdf.set_index("rid").label.loc[r.index]
    print(f"[{nm}] REVIEW vs star rating      : acc={accuracy_score(gold,r):.4f} "
          f"macroF1={f1_score(gold,r,average='macro',labels=LABELS):.4f}")

s_h = en[en.split == "hold"]
r_h = rollup(s_h, model.predict(s_h.phrase))
gold_h = ho_r.set_index("rid").label.loc[r_h.index]
print()
print(classification_report(gold_h, r_h, labels=LABELS, digits=4))

[calib] sentence vs machine labels: acc=0.8426


[calib] REVIEW vs star rating      : acc=0.6199 macroF1=0.5500


[hold] sentence vs machine labels: acc=0.8191


[hold] REVIEW vs star rating      : acc=0.5910 macroF1=0.5227



              precision    recall  f1-score   support

    negative     0.5714    0.4311    0.4915       167
     neutral     0.2707    0.3451    0.3034       142
    positive     0.7708    0.7754    0.7731       334

    accuracy                         0.5910       643
   macro avg     0.5377    0.5172    0.5227       643
weighted avg     0.6086    0.5910    0.5962       643



## The probe that caught the original bug

Kept as a permanent regression check. These are clean, unambiguous sentences
of the kind a user will actually type — none of them resembles the fragmented
training text, which is exactly the point.

In [5]:
PROBE = [
    ("positive", "The room was spotless and the staff could not have been more helpful."),
    ("positive", "Excellent breakfast, lovely view, we will definitely come back."),
    ("positive", "Beautiful hotel, very comfortable bed, great value for money."),
    ("negative", "Freezing cold room, the heating never worked and nobody came to fix it."),
    ("negative", "Dirty bathroom, rude reception, terrible experience overall."),
    ("negative", "Worst hotel I have ever stayed in. Avoid at all costs."),
    ("neutral",  "Arrived late, checked in, went to sleep. Nothing special."),
    ("neutral",  "The hotel is near the station. Rooms are standard."),
]
pred = model.predict([normalize(t) for _, t in PROBE])
ok = sum(p == y for (y, _), p in zip(PROBE, pred))
print(f"probe: {ok}/{len(PROBE)}\n")
for (y, t), p in zip(PROBE, pred):
    print(f"  {'ok ' if p==y else 'XX '}{y:<8} -> {p:<8} | {t[:64]}")
print()
print("The remaining failure is instructive: 'could not have been more helpful'")
print("is positive stated through a negation, and a bag-of-ngrams model has no")
print("way to scope 'not'. That is a structural limit of the backbone, not a")
print("labelling problem -- and it is the clearest argument for a transformer.")

probe: 7/8

  XX positive -> negative | The room was spotless and the staff could not have been more hel
  ok positive -> positive | Excellent breakfast, lovely view, we will definitely come back.
  ok positive -> positive | Beautiful hotel, very comfortable bed, great value for money.
  ok negative -> negative | Freezing cold room, the heating never worked and nobody came to 
  ok negative -> negative | Dirty bathroom, rude reception, terrible experience overall.
  ok negative -> negative | Worst hotel I have ever stayed in. Avoid at all costs.
  ok neutral  -> neutral  | Arrived late, checked in, went to sleep. Nothing special.
  ok neutral  -> neutral  | The hotel is near the station. Rooms are standard.

The remaining failure is instructive: 'could not have been more helpful'
is positive stated through a negation, and a bag-of-ngrams model has no
way to scope 'not'. That is a structural limit of the backbone, not a
labelling problem -- and it is the clearest argument for a transfor

In [6]:
mp = OUT / "model_en_sentence.joblib"
joblib.dump(model, mp, compress=3)
en[["rid","phrase","sentiment","aspect_label","note","split","hotel_name"]].to_csv(
    OUT / "sentences.csv", index=False)
for nm, d in [("train", tr_r), ("calib", ca_r), ("hold", ho_r), ("test", te_r)]:
    d.to_csv(OUT / f"reviews_{nm}.csv", index=False)

json.dump({"unit": "sentence", "rollup": "majority vote per review",
           "sentences_train": int(len(s_tr)),
           "review_holdout": {"accuracy": round(float(accuracy_score(gold_h, r_h)), 4),
                              "macro_f1": round(float(f1_score(gold_h, r_h,
                                   average="macro", labels=LABELS)), 4)},
           "probe": f"{ok}/{len(PROBE)}",
           "superseded": {"review_level_star_labels":
                          {"macro_f1": 0.6258, "probe": "6/8"}},
           "size_mb": round(mp.stat().st_size/1e6, 2)},
          open(OUT / "results_sentence_model.json", "w"), indent=2)
print(f"saved {mp.name} ({mp.stat().st_size/1e6:.2f} MB) + sentences.csv")

saved model_en_sentence.joblib (0.64 MB) + sentences.csv
